# ATLAS-ML: Exploratory Data Analysis

Exploring the datasets behind the project: 56 Moon-base / Earth-site pairs and 14 Earth site profiles. Questions: How are the Analog Scores distributed? Which features are missing, and why? Which pillars correlate with the score?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

scores = pd.read_csv('../data/scores_ranking.csv')
earth = pd.read_csv('../data/earth_profiles.csv')
moon = pd.read_csv('../data/moon_profiles.csv')
print('pairs:', scores.shape, '| earth sites:', earth.shape, '| moon bases:', moon.shape)
scores.head()

## Missing values
`dist_soil` is missing for 12 pairs. Those are exactly the pairs involving the two Antarctic sites and Etosha Pan: SoilGrids has no soil coverage there. This is *informative* missingness, so the pipeline median-imputes it and adds a `soil_missing` flag.

In [ ]:
scores.isna().sum().sort_values(ascending=False).head(8)

In [ ]:
scores[scores['dist_soil'].isna()]['earth_site'].unique()

## Target distribution: the Analog Score

In [ ]:
scores['analog_score'].describe()

In [ ]:
scores['analog_score'].hist(bins=12, color='#2b6cb0')
plt.xlabel('Analog Score'); plt.ylabel('Pairs'); plt.title('Distribution of Analog Scores (56 pairs)'); plt.show()

## Which pillars move with the score?
Correlation of each pillar distance with the Analog Score. Expect strong negative correlations: a bigger distance means a worse match.

In [ ]:
pillars = ['dist_slope','dist_flat','dist_temperature','dist_relief','dist_soil']
scores[pillars + ['analog_score']].corr()['analog_score'].drop('analog_score').sort_values()

## Best match per base

In [ ]:
scores.loc[scores.groupby('moon_base')['analog_score'].idxmax(), ['moon_base','earth_site','analog_score']]

## Findings
1. Scores range 40-72, mean ~55: no Earth site is a perfect lunar stand-in, temperature is the hardest pillar to match.
2. Missing soil data is geographic (polar / no-coverage regions), not random, so it is flagged, not just filled.
3. The pair dataset (56 rows x 5 distance features) is small but complete enough for cross-validated model comparison, which is what the model scripts do.